# Multi-class Text Classification with BERT

**Goal:** Fine-tune a compact BERT model to classify news articles into four AG News topics.

This notebook is intentionally CPU-friendly. It uses a small subset of the dataset and a compact BERT checkpoint so the project can be demonstrated on a computer without a GPU.

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch

from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.14.0+cpu
Device: cpu


## 1. Download the dataset

AG News is a public four-class news topic dataset. The CSV is downloaded only when it is missing, which keeps the Git repository small.

In [2]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
DATA_PATH = DATA_DIR / "ag_news.csv"

if not DATA_PATH.exists():
    response = requests.get(DATA_URL, timeout=30)
    response.raise_for_status()
    DATA_PATH.write_bytes(response.content)
    print(f"Downloaded dataset to {DATA_PATH}")
else:
    print(f"Using existing dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, header=None, names=["label", "title", "description"])
df["text"] = (df["title"].fillna("") + ". " + df["description"].fillna("")).str.strip()

# AG News labels in this CSV are 1-4.
label_names = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech",
}
df["label_name"] = df["label"].map(label_names)
df.head()

Using existing dataset: ..\data\ag_news.csv


,label,title,description,text,label_name
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,Business
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,Business
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,Business
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...,Business
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new...",Business


## 2. Create a small CPU-friendly experiment

A full AG News fine-tuning run is unnecessary for demonstrating the workflow. We start with a small stratified sample and keep the configuration easy to increase later.

In [3]:
TRAIN_SAMPLES = 800
TEST_SAMPLES = 200

parts = []
per_class_train = TRAIN_SAMPLES // 4
per_class_test = TEST_SAMPLES // 4

for label in sorted(label_names):
    class_df = df[df["label"] == label].sample(
        n=per_class_train + per_class_test,
        random_state=SEED
    )
    parts.append(class_df)

small_df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_parts = []
test_parts = []

for label in sorted(label_names):
    class_df = small_df[small_df["label"] == label]
    train_parts.append(class_df.iloc[:per_class_train])
    test_parts.append(class_df.iloc[per_class_train:per_class_train + per_class_test])

train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Convert labels to zero-based IDs for Transformers.
train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print(train_df["label"].value_counts().sort_index())

Training rows: 800
Testing rows: 200
label
0    200
1    200
2    200
3    200
Name: count, dtype: int64


## 3. Load a compact BERT checkpoint

`google/bert_uncased_L-2_H-128_A-2` is a small BERT model designed for experimentation. It is much lighter than a standard BERT-base model and is more practical for CPU-only learning.

In [4]:
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = train_df["text"].tolist()
test_texts = test_df["text"].tolist()

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=64,
)
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=64,
)

In [5]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = NewsDataset(train_encodings, train_df["label"].tolist())
test_dataset = NewsDataset(test_encodings, test_df["label"].tolist())

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"},
    label2id={"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3},
)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 4. Fine-tune BERT

The settings below intentionally favor low memory usage. On a CPU, the first run may still take some time.

In [6]:
import sys
import subprocess

# Install into the same Python interpreter as this notebook kernel.
try:
    import accelerate
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "accelerate>=1.1.0"]
    )
    import accelerate


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}


output_dir = Path("../models/bert_agnews")

training_args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    use_cpu=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.296542,1.279590,0.605000


TrainOutput(global_step=200, training_loss=1.3313008880615234, metrics={'train_runtime': 31.2871, 'train_samples_per_second': 25.57, 'train_steps_per_second': 6.392, 'total_flos': 127127961600.0, 'train_loss': 1.3313008880615234, 'epoch': 1.0})

## 5. Evaluate the classifier

In [7]:
evaluation = trainer.evaluate()
print("Evaluation:", evaluation)

prediction_output = trainer.predict(test_dataset)
predictions = np.argmax(prediction_output.predictions, axis=1)

print(classification_report(
    test_df["label"],
    predictions,
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    zero_division=0,
))

Training Loss,Validation Loss,Epoch,Accuracy
1.296542,1.279590,1,0.605000


Evaluation: {'eval_loss': 1.2795897722244263, 'eval_accuracy': 0.605}


              precision    recall  f1-score   support

       World       0.69      0.68      0.69        50
      Sports       0.85      0.78      0.81        50
    Business       0.43      0.68      0.53        50
    Sci/Tech       0.54      0.28      0.37        50

    accuracy                           0.60       200
   macro avg       0.63      0.60      0.60       200
weighted avg       0.63      0.60      0.60       200



## 6. Save the fine-tuned model

Saving the model locally makes it possible for the Flask application to use the trained classifier without retraining every time.

In [8]:
output_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

print(f"Model saved to: {output_dir.resolve()}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: C:\Users\user\Desktop\Multi-class-Text-Classification\models\bert_agnews


## 7. Test the model on new text

In [9]:
def predict_topic(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64,
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_id = int(torch.argmax(outputs.logits, dim=1).item())
    return model.config.id2label[predicted_id]

examples = [
    "The national team won the championship after a dramatic final.",
    "The company reported higher quarterly profits after strong sales.",
    "Scientists announced a new telescope mission to study distant galaxies.",
    "World leaders met to discuss a new international agreement.",
]

for text in examples:
    print(f"{predict_topic(text):10s} | {text}")

Sports     | The national team won the championship after a dramatic final.
Business   | The company reported higher quarterly profits after strong sales.
Sci/Tech   | Scientists announced a new telescope mission to study distant galaxies.
Sports     | World leaders met to discuss a new international agreement.


## 8. Next step: Flask deployment

Once the final cell has saved `models/bert_agnews/`, run this command from the repository root:

```bash
python app.py
```

Then open `http://127.0.0.1:5000`.

The web application loads the saved tokenizer and BERT classifier and provides a simple interface for live predictions.